Please run every cell in order. This notebook requires a GPU: go to Runtime → Change runtime type, select GPU, and then rerun the cells.

In [ ]:
!pip install speechbrain gradio gTTS torch torchaudio --quiet


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 864.1/864.1 kB 30.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 98.2/98.2 kB 6.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 119.9/119.9 kB 7.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 788.2/788.2 kB 40.7 MB/s eta 0:00:00


In [ ]:
"""
Emotion-Aware Voice Reminder App (Google Colab Version)
-------------------------------------------------------

- Upload or record a short voice clip.
- Detect emotion using SpeechBrain (IEMOCAP model).
- Map to 4 tones: sad, happy, neutral, excited.
- Generate a voice reminder in that tone using gTTS.

Run this cell, then open the Gradio link that appears.
"""

import os
import uuid
from typing import Tuple, Optional

import gradio as gr
from gtts import gTTS
from speechbrain.inference.interfaces import foreign_class


# ---------------------------------------------------------------------------
# 1. Emotion detection using SpeechBrain (IEMOCAP model)
# ---------------------------------------------------------------------------

class EmotionDetector:
    """
    Wraps SpeechBrain's pretrained emotion recognition model:
    https://huggingface.co/speechbrain/emotion-recognition-wav2vec2-IEMOCAP
    """

    def __init__(self, device: str = "cpu"):
        """
        :param device: "cpu" or "cuda" if you have a GPU
        """
        self.device = device
        self.classifier = foreign_class(
            source="speechbrain/emotion-recognition-wav2vec2-IEMOCAP",
            pymodule_file="custom_interface.py",
            classname="CustomEncoderWav2vec2Classifier",
            run_opts={"device": self.device},
        )

        # Raw labels (from the model) -> high-level tones
        self.label_to_tone = {
            "angry": "sad",
            "sad": "sad",
            "frustrated": "sad",
            "fear": "sad",
            "disappointed": "sad",

            "happy": "happy",

            "excited": "excited",
            "surprised": "excited",

            "neutral": "neutral",
        }

    def predict(self, audio_path: str) -> Tuple[str, float, str]:
        """
        Run emotion detection on the given audio file.

        :param audio_path: path to wav/mp3/etc.
        :return: (raw_emotion_label, confidence_score, mapped_tone)
        """
        out_prob, score, index, text_lab = self.classifier.classify_file(audio_path)

        # text_lab can be a list-like; convert to plain string
        if isinstance(text_lab, (list, tuple)):
            raw_label = str(text_lab[0])
        else:
            raw_label = str(text_lab)

        # Normalize score to float
        if isinstance(score, (list, tuple)):
            conf = float(score[0])
        else:
            try:
                conf = float(score)
            except Exception:
                conf = 0.0

        mapped_tone = self._map_label_to_tone(raw_label)

        return raw_label, conf, mapped_tone

    def _map_label_to_tone(self, label: str) -> str:
        label = label.lower().strip()
        return self.label_to_tone.get(label, "neutral")


# ---------------------------------------------------------------------------
# 2. Voice reminder generator using gTTS (works in Colab)
# ---------------------------------------------------------------------------

class VoiceReminder:
    """
    Uses gTTS to generate a TTS reminder file.
    Tone control is approximated by adjusting speed (slow for sad).
    """

    DEFAULT_REMINDERS = {
        "sad": (
            "Hey, I hear some heaviness in your voice. "
            "Take a deep breath and remember to be kind to yourself."
        ),
        "happy": (
            "You sound cheerful! Keep that energy going "
            "and remember to share your happiness with others."
        ),
        "neutral": (
            "Thanks for checking in. Stay focused, stay calm, "
            "and keep moving forward one step at a time."
        ),
        "excited": (
            "You sound excited! Channel that energy into "
            "something amazing today."
        ),
    }

    def __init__(self, output_dir: str = "reminders"):
        self.output_dir = output_dir
        os.makedirs(self.output_dir, exist_ok=True)

    def build_text(self, tone: str, custom_text: Optional[str]) -> str:
        """
        Decide which text to speak: custom one or default for that tone.
        """
        tone = (tone or "neutral").lower()
        custom_text = (custom_text or "").strip()

        if custom_text:
            return custom_text

        return self.DEFAULT_REMINDERS.get(tone, self.DEFAULT_REMINDERS["neutral"])

    def speak_to_file(self, text: str, tone: str) -> str:
        """
        Generate a reminder audio file using gTTS and return its path.
        """
        tone = (tone or "neutral").lower()

        # Approximate tone control via speed
        slow = tone == "sad"

        tts = gTTS(text=text, lang="en", slow=slow)
        filename = f"reminder_{tone}_{uuid.uuid4().hex[:8]}.mp3"
        out_path = os.path.join(self.output_dir, filename)
        tts.save(out_path)

        return out_path


# ---------------------------------------------------------------------------
# 3. Glue logic for Gradio
# ---------------------------------------------------------------------------

emotion_detector = EmotionDetector(device="cpu")  # "cuda" if you have GPU in Colab
voice_reminder = VoiceReminder()


def analyze_and_remind(audio_path: str, custom_text: str):
    """
    Gradio callback:
    - audio_path: path to an uploaded audio file
    - custom_text: optional custom reminder text
    Returns:
        - raw_emotion
        - confidence
        - mapped_tone
        - final_reminder_text
        - reminder_audio_path
    """
    if not audio_path:
        raise gr.Error("Please upload or record an audio clip first.")

    # 1) Detect emotion
    raw_label, conf, tone = emotion_detector.predict(audio_path)

    # 2) Build reminder text (custom or default)
    reminder_text = voice_reminder.build_text(tone, custom_text)

    # 3) Generate voice reminder audio
    reminder_audio_path = voice_reminder.speak_to_file(reminder_text, tone)

    # Nicely formatted strings for display
    raw_label_display = raw_label
    confidence_display = f"{conf:.2f}"
    tone_display = tone.capitalize()

    return (
        raw_label_display,
        confidence_display,
        tone_display,
        reminder_text,
        reminder_audio_path,
    )


# ---------------------------------------------------------------------------
# 4. Gradio UI
# ---------------------------------------------------------------------------

def build_interface() -> gr.Blocks:
    with gr.Blocks(title="Emotion-Aware Voice Reminder (Colab)") as demo:
        gr.Markdown(
            """
        # 🎙️ Emotion-Aware Voice Reminder

        1. Upload or record a short voice clip.
        2. The app detects the underlying emotion.
        3. It maps it to: **Sad, Happy, Neutral, Excited**.
        4. You get a **voice reminder** spoken back in that tone.
        """
        )

        with gr.Row():
            with gr.Column():
                audio_input = gr.Audio(
                    sources=["microphone", "upload"],
                    type="filepath",
                    label="Step 1 – Upload / record your voice",
                )
                custom_text = gr.Textbox(
                    label="Step 2 – Optional: custom reminder text",
                    placeholder=(
                        "Write your own reminder message here... "
                        "Leave blank to use a tone-specific default."
                    ),
                    lines=3,
                )
                run_btn = gr.Button("✨ Detect Emotion & Generate Reminder")

            with gr.Column():
                raw_emotion = gr.Textbox(
                    label="Detected raw emotion (model output)",
                    interactive=False,
                )
                confidence = gr.Textbox(
                    label="Model confidence",
                    interactive=False,
                )
                mapped_tone = gr.Textbox(
                    label="Mapped tone (Sad / Happy / Neutral / Excited)",
                    interactive=False,
                )
                final_text = gr.Textbox(
                    label="Reminder text that was spoken",
                    interactive=False,
                    lines=4,
                )
                reminder_audio = gr.Audio(
                    label="Voice reminder (play or download)",
                    type="filepath",
                )

        run_btn.click(
            fn=analyze_and_remind,
            inputs=[audio_input, custom_text],
            outputs=[raw_emotion, confidence, mapped_tone, final_text, reminder_audio],
        )

        return demo


demo = build_interface()
demo.launch(share=True)


/usr/local/lib/python3.12/dist-packages/speechbrain/utils/torch_audio_backend.py:57: UserWarning: torchaudio._backend.list_audio_backends has been deprecated. This deprecation is part of a large refactoring effort to transition TorchAudio into a maintenance phase. The decoding and encoding capabilities of PyTorch for both audio and video are being consolidated into TorchCodec. Please see https://github.com/pytorch/audio/issues/3902 for more information. It will be removed from the 2.9 release. 
  available_backends = torchaudio.list_audio_backends()
DEBUG:speechbrain.utils.checkpoints:Registered checkpoint save hook for _speechbrain_save
DEBUG:speechbrain.utils.checkpoints:Registered checkpoint load hook for _speechbrain_load
DEBUG:speechbrain.utils.checkpoints:Registered checkpoint save hook for save
DEBUG:speechbrain.utils.checkpoints:Registered checkpoint load hook for load
DEBUG:speechbrain.utils.checkpoints:Registered checkpoint save hook for _save
DEBUG:speechbrain.utils.checkpoi

hyperparams.yaml: 0.00B [00:00, ?B/s]

INFO:speechbrain.utils.fetching:Fetch custom_interface.py: Fetching from HuggingFace Hub 'speechbrain/emotion-recognition-wav2vec2-IEMOCAP' if not cached


custom_interface.py: 0.00B [00:00, ?B/s]

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


...

model.ckpt:   0%|          | 0.00/13.2k [00:00<?, ?B/s]

DEBUG:speechbrain.utils.parameter_transfer:Set local path in self.paths["model"] = /root/.cache/huggingface/hub/models--speechbrain--emotion-recognition-wav2vec2-IEMOCAP/snapshots/117a9c3dff08be81a3628eecf6a66b547ec1659b/model.ckpt
INFO:speechbrain.utils.fetching:Fetch label_encoder.txt: Fetching from HuggingFace Hub 'speechbrain/emotion-recognition-wav2vec2-IEMOCAP' if not cached


label_encoder.txt:   0%|          | 0.00/83.0 [00:00<?, ?B/s]

DEBUG:speechbrain.utils.parameter_transfer:Set local path in self.paths["label_encoder"] = /root/.cache/huggingface/hub/models--speechbrain--emotion-recognition-wav2vec2-IEMOCAP/snapshots/117a9c3dff08be81a3628eecf6a66b547ec1659b/label_encoder.txt
INFO:speechbrain.utils.parameter_transfer:Loading pretrained files for: wav2vec2, model, label_encoder
DEBUG:speechbrain.utils.parameter_transfer:Redirecting (loading from local path): wav2vec2 -> /root/.cache/huggingface/hub/models--speechbrain--emotion-recognition-wav2vec2-IEMOCAP/snapshots/117a9c3dff08be81a3628eecf6a66b547ec1659b/wav2vec2.ckpt
DEBUG:speechbrain.utils.parameter_transfer:Redirecting (loading from local path): model -> /root/.cache/huggingface/hub/models--speechbrain--emotion-recognition-wav2vec2-IEMOCAP/snapshots/117a9c3dff08be81a3628eecf6a66b547ec1659b/model.ckpt
DEBUG:speechbrain.utils.parameter_transfer:Redirecting (loading from local path): label_encoder -> /root/.cache/huggingface/hub/models--speechbrain--emotion-recogni

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://1069fece0d505233d4.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [ ]:
!pip install --quiet \
  speechbrain \
  gradio \
  coqui-tts \
  torch \
  torchaudio


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.7/3.7 MB 62.7 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 85.3/85.3 kB 9.2 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.0/42.0 kB 3.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 101.6/101.6 kB 10.9 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.3/15.3 MB 94.6 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.1/18.1 MB 67.9 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 31.4/31.4 MB 12.7 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.9/10.9 MB 70.1 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━

In [ ]:
"""
Emotion-Aware Voice Cloned Reminder (Colab + GPU)
-------------------------------------------------

Pipeline:
1. User uploads/records a short audio clip with their own voice.
2. We run SpeechBrain emotion recognition to detect the emotion.
3. We map raw emotion -> [sad, happy, neutral, excited].
4. We build a reminder text based on that tone (or a custom text you type).
5. We use Coqui XTTS-v2 (voice cloning TTS) to speak the reminder
   in the *same* voice as the uploaded audio.

NOTE: XTTS-v2 is non-commercial and should only be used with voices
you have rights/consent to use.
"""

import os
import uuid
from typing import Tuple, Optional

import torch
import gradio as gr
from speechbrain.inference.interfaces import foreign_class
from TTS.api import TTS


# ============================================================
# 1. Emotion detection with SpeechBrain (IEMOCAP model)
# ============================================================

class EmotionDetector:
    """
    Uses: speechbrain/emotion-recognition-wav2vec2-IEMOCAP
    Docs / example: see Hugging Face model card.
    """

    def __init__(self, device: str = "cpu"):
        self.device = device
        # This will auto-download model + custom_interface.py from HF
        self.classifier = foreign_class(
            source="speechbrain/emotion-recognition-wav2vec2-IEMOCAP",
            pymodule_file="custom_interface.py",
            classname="CustomEncoderWav2vec2Classifier",
            run_opts={"device": self.device},
        )

        # Map raw labels from the model to a small set of tones
        self.label_to_tone = {
            # negative
            "angry": "sad",
            "frustrated": "sad",
            "sad": "sad",
            "fear": "sad",
            "disappointed": "sad",

            # positive
            "happy": "happy",

            # high arousal positive
            "excited": "excited",
            "surprised": "excited",

            # baseline
            "neutral": "neutral",
        }

    def predict(self, audio_path: str) -> Tuple[str, float, str]:
        """
        :param audio_path: path to uploaded audio file
        :return: (raw_emotion_label, confidence, mapped_tone)
        """
        out_prob, score, index, text_lab = self.classifier.classify_file(audio_path)

        # text_lab is usually a list-like tensor of labels, e.g. ('sad',)
        if isinstance(text_lab, (list, tuple)):
            raw_label = str(text_lab[0])
        else:
            raw_label = str(text_lab)

        # convert score to float
        if isinstance(score, (list, tuple)):
            conf = float(score[0])
        else:
            try:
                conf = float(score)
            except Exception:
                conf = 0.0

        mapped_tone = self._map_label_to_tone(raw_label)

        return raw_label, conf, mapped_tone

    def _map_label_to_tone(self, label: str) -> str:
        label = label.lower().strip()
        return self.label_to_tone.get(label, "neutral")


# ============================================================
# 2. Voice cloning TTS with Coqui XTTS-v2
# ============================================================

class VoiceCloner:
    """
    Wraps Coqui XTTS-v2 via the coqui-tts API.

    - We use the uploaded audio as `speaker_wav` so the generated
      reminder is in *your* voice.
    """

    DEFAULT_REMINDERS = {
        "sad": (
            "Hey, I can hear you might be feeling low. "
            "Please slow down, take a breath, and give yourself some kindness today."
        ),
        "happy": (
            "Nice energy! Keep that smile going and remember to enjoy the little moments today."
        ),
        "neutral": (
            "Stay calm and focused. One step at a time, you are moving in the right direction."
        ),
        "excited": (
            "You sound full of energy! Use that excitement to do something awesome today."
        ),
    }

    def __init__(self, device: str = "cpu", out_dir: str = "cloned_reminders"):
        self.device = device
        self.out_dir = out_dir
        os.makedirs(self.out_dir, exist_ok=True)

        # Initialize XTTS-v2 (multi-speaker, multilingual, voice cloning)
        # Model docs: tts_models/multilingual/multi-dataset/xtts_v2
        print("🔊 Loading XTTS-v2 voice cloning model (this can take a bit)...")
        self.tts = TTS("tts_models/multilingual/multi-dataset/xtts_v2").to(self.device)
        print("✅ XTTS-v2 loaded.")

    def build_text(self, tone: str, custom_text: Optional[str]) -> str:
        tone = (tone or "neutral").lower()
        custom_text = (custom_text or "").strip()

        if custom_text:
            return custom_text

        return self.DEFAULT_REMINDERS.get(tone, self.DEFAULT_REMINDERS["neutral"])

    def synthesize(
        self,
        text: str,
        speaker_ref_path: str,
        tone: str,
        language: str = "en",
    ) -> str:
        """
        Generate speech in cloned voice.

        :param text: text to speak
        :param speaker_ref_path: path to user's reference audio (your voice)
        :param tone: sad | happy | neutral | excited (semantic, affects text only)
        :param language: ISO code ("en", "hi", "ta", etc., XTTS supports 17+ langs)
        :return: path to generated wav file
        """
        # NOTE: XTTS handles resampling & speaker embedding internally
        filename = f"reminder_{tone}_{uuid.uuid4().hex[:8]}.wav"
        out_path = os.path.join(self.out_dir, filename)

        # We pass the reference clip as speaker_wav so it clones that voice
        # For better results, use a clean 6–20 second clip of you speaking.
        self.tts.tts_to_file(
            text=text,
            file_path=out_path,
            speaker_wav=[speaker_ref_path],
            language=language,
        )

        return out_path


# ============================================================
# 3. Glue: end-to-end function for Gradio
# ============================================================

# Choose device
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", DEVICE)

emotion_detector = EmotionDetector(device=DEVICE)
voice_cloner = VoiceCloner(device=DEVICE)


def analyze_and_clone_voice(audio_path: str,
                            custom_text: str,
                            language: str):
    """
    Gradio callback:
    1. Detect emotion from uploaded/recorded audio.
    2. Map emotion to 4-tone set.
    3. Build reminder text (custom or default).
    4. Use XTTS-v2 to speak the reminder in the *same* voice.

    Returns:
      - raw_emotion_label
      - confidence
      - mapped_tone
      - final_reminder_text
      - generated_audio_path
    """
    if not audio_path:
        raise gr.Error("Please upload or record an audio clip of YOUR voice first.")

    # --- Emotion detection ---
    raw_label, conf, tone = emotion_detector.predict(audio_path)

    # --- Build text ---
    reminder_text = voice_cloner.build_text(tone, custom_text)

    # --- Voice cloning TTS ---
    # NOTE: We reuse the same audio as the speaker reference.
    generated_path = voice_cloner.synthesize(
        text=reminder_text,
        speaker_ref_path=audio_path,
        tone=tone,
        language=language or "en",
    )

    # Human-friendly strings
    raw_label_display = raw_label
    confidence_display = f"{conf:.2f}"
    tone_display = tone.capitalize()

    return (
        raw_label_display,
        confidence_display,
        tone_display,
        reminder_text,
        generated_path,
    )


# ============================================================
# 4. Gradio interface
# ============================================================

def build_interface():
    with gr.Blocks(title="Emotion-Aware Voice Cloned Reminder") as demo:
        gr.Markdown(
            """
        # 🎙️ Emotion-Aware Voice Cloned Reminder

        - Upload or record **your own voice** (5–20 seconds is ideal).
        - The model detects your **emotion**.
        - It maps it to one of: **Sad, Happy, Neutral, Excited**.
        - It generates a **reminder sentence** (or uses your custom text).
        - Then XTTS-v2 **clones your voice** and speaks that reminder.

        ⚠️ Use this only with voices you have rights & consent for.
        """
        )

        with gr.Row():
            with gr.Column():
                audio_in = gr.Audio(
                    sources=["microphone", "upload"],
                    type="filepath",
                    label="Step 1 – Upload / record YOUR voice",
                )

                custom_text = gr.Textbox(
                    label="Step 2 – Optional: custom reminder text",
                    placeholder="Leave empty to auto-generate tone-based reminder...",
                    lines=3,
                )

                language = gr.Dropdown(
                    choices=["en", "es", "fr", "de", "it", "pt", "pl", "tr", "ru", "nl", "cs", "ar", "zh", "ja", "ko", "hi"],
                    value="en",
                    label="Language for the reminder (XTTS-v2 supports multiple)",
                )

                run_btn = gr.Button("✨ Detect Emotion + Clone My Voice")

            with gr.Column():
                raw_emotion = gr.Textbox(
                    label="Detected raw emotion (model output)",
                    interactive=False,
                )
                confidence = gr.Textbox(
                    label="Emotion model confidence",
                    interactive=False,
                )
                mapped_tone = gr.Textbox(
                    label="Mapped tone (Sad / Happy / Neutral / Excited)",
                    interactive=False,
                )
                final_text = gr.Textbox(
                    label="Reminder text that was spoken",
                    interactive=False,
                    lines=4,
                )
                out_audio = gr.Audio(
                    label="Cloned-voice reminder (play / download)",
                    type="filepath",
                )

        run_btn.click(
            fn=analyze_and_clone_voice,
            inputs=[audio_in, custom_text, language],
            outputs=[raw_emotion, confidence, mapped_tone, final_text, out_audio],
        )

        return demo


demo = build_interface()
demo.launch(share=True)


/usr/local/lib/python3.12/dist-packages/speechbrain/utils/torch_audio_backend.py:57: UserWarning: torchaudio._backend.list_audio_backends has been deprecated. This deprecation is part of a large refactoring effort to transition TorchAudio into a maintenance phase. The decoding and encoding capabilities of PyTorch for both audio and video are being consolidated into TorchCodec. Please see https://github.com/pytorch/audio/issues/3902 for more information. It will be removed from the 2.9 release. 
  available_backends = torchaudio.list_audio_backends()
DEBUG:speechbrain.utils.checkpoints:Registered checkpoint save hook for _speechbrain_save
DEBUG:speechbrain.utils.checkpoints:Registered checkpoint load hook for _speechbrain_load
DEBUG:speechbrain.utils.checkpoints:Registered checkpoint save hook for save
DEBUG:speechbrain.utils.checkpoints:Registered checkpoint load hook for load
DEBUG:speechbrain.utils.checkpoints:Registered checkpoint save hook for _save
DEBUG:speechbrain.utils.checkpoi

Using device: cuda


hyperparams.yaml: 0.00B [00:00, ?B/s]

INFO:speechbrain.utils.fetching:Fetch custom_interface.py: Fetching from HuggingFace Hub 'speechbrain/emotion-recognition-wav2vec2-IEMOCAP' if not cached


custom_interface.py: 0.00B [00:00, ?B/s]

...

label_encoder.txt:   0%|          | 0.00/83.0 [00:00<?, ?B/s]

DEBUG:speechbrain.utils.parameter_transfer:Set local path in self.paths["label_encoder"] = /root/.cache/huggingface/hub/models--speechbrain--emotion-recognition-wav2vec2-IEMOCAP/snapshots/117a9c3dff08be81a3628eecf6a66b547ec1659b/label_encoder.txt
INFO:speechbrain.utils.parameter_transfer:Loading pretrained files for: wav2vec2, model, label_encoder
DEBUG:speechbrain.utils.parameter_transfer:Redirecting (loading from local path): wav2vec2 -> /root/.cache/huggingface/hub/models--speechbrain--emotion-recognition-wav2vec2-IEMOCAP/snapshots/117a9c3dff08be81a3628eecf6a66b547ec1659b/wav2vec2.ckpt
DEBUG:speechbrain.utils.parameter_transfer:Redirecting (loading from local path): model -> /root/.cache/huggingface/hub/models--speechbrain--emotion-recognition-wav2vec2-IEMOCAP/snapshots/117a9c3dff08be81a3628eecf6a66b547ec1659b/model.ckpt
DEBUG:speechbrain.utils.parameter_transfer:Redirecting (loading from local path): label_encoder -> /root/.cache/huggingface/hub/models--speechbrain--emotion-recogni

🔊 Loading XTTS-v2 voice cloning model (this can take a bit)...
 > You must confirm the following:
 | > "I have purchased a commercial license from Coqui: licensing@coqui.ai"
 | > "Otherwise, I agree to the terms of the non-commercial CPML: https://coqui.ai/cpml" - [y/n]
 | | > y


100%|██████████| 1.87G/1.87G [00:22<00:00, 82.4MiB/s]
4.37kiB [00:00, 8.88MiB/s]
361kiB [00:00, 62.3MiB/s]
100%|██████████| 32.0/32.0 [00:00<00:00, 64.0kiB/s]
100%|██████████| 7.75M/7.75M [00:00<00:00, 82.0MiB/s]


✅ XTTS-v2 loaded.
Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://37ae5c5bac02f6e270.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
